# Statistical Market Analysis For Used Vehicles

## Introduction

Sharma Auto Ventures is planning to enter the Indian automobile market as a vehicle dealership and distribution company. As a new participant in a highly competitive industry, the company seeks to understand customer preferences, market trends, and the factors that influence vehicle demand across India. Gaining these insights will enable the company to make informed investment decisions and build an inventory that aligns with market needs.

The primary question guiding this analysis is:

**Which types of vehicles are most preferred by consumers in the Indian market, and what vehicle characteristics should Sharma Auto Ventures prioritize when selecting inventory?**

To answer this question, used-car market data will be analyzed to identify patterns in vehicle pricing, fuel type, transmission type, vehicle age, mileage, ownership history, and brand popularity. Exploratory data analysis and data visualization techniques will be used to uncover trends and relationships that influence customer purchasing decisions.

The findings from this analysis will help Sharma Auto Ventures develop a data-driven inventory strategy, identify high-demand vehicle categories, and make better pricing decisions. Ultimately, the insights will support the company's goal of establishing a strong presence in the Indian automobile market while maximizing profitability and customer satisfaction.


# Imports cell

In [29]:
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

In [30]:
pd.set_option("display.max_columns", None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
df = pd.read_csv("used_car_price_prediction_1M.csv")
df.head()

,Brand,Model,Year,Mileage_kmpl,Engine_CC,Horsepower,Fuel_Type,Transmission,Owner_Type,Color,City,Kms_Driven,Insurance_Valid,Service_History,Accidents,Tax_Paid,Number_of_Doors,Seats,Registration_Age,Price
0,Toyota,Camry,2001,19.616,1396.560,NaN,Hybrid,Automatic,First,Grey,Chennai,500928,1,1,0,1,4,5,24,353189.604
1,Hyundai,i20,2012,19.479,1130.771,137.022,Petrol,NaN,First,Grey,Mumbai,211510,0,1,0,1,4,5,13,959694.257
2,Mahindra,Bolero,2008,17.470,1766.466,142.903,NaN,Automatic,First,Black,Kolkata,333999,1,0,0,1,5,5,17,292022.926
3,BMW,X1,2015,21.295,NaN,49.782,NaN,Manual,First,White,Pune,225930,1,1,3,1,4,5,10,2949453.462
4,Honda,Civic,2022,18.327,1239.335,121.506,Diesel,Manual,Second,White,Kolkata,43404,1,0,0,1,5,5,3,636520.798


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1005000 entries, 0 to 1004999
Data columns (total 20 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   Brand             1005000 non-null  object 
 1   Model             1005000 non-null  object 
 2   Year              1005000 non-null  int64  
 3   Mileage_kmpl      924610 non-null   float64
 4   Engine_CC         924597 non-null   float64
 5   Horsepower        924597 non-null   float64
 6   Fuel_Type         926267 non-null   object 
 7   Transmission      924615 non-null   object 
 8   Owner_Type        1005000 non-null  object 
 9   Color             924581 non-null   object 
 10  City              924570 non-null   object 
 11  Kms_Driven        1005000 non-null  int64  
 12  Insurance_Valid   1005000 non-null  int64  
 13  Service_History   1005000 non-null  int64  
 14  Accidents         1005000 non-null  int64  
 15  Tax_Paid          1005000 non-null  int64  
 16  

In [32]:
print(df.isnull().sum() / len(df) * 100)

Brand              0.000
Model              0.000
Year               0.000
Mileage_kmpl       7.999
Engine_CC          8.000
Horsepower         8.000
Fuel_Type          7.834
Transmission       7.999
Owner_Type         0.000
Color              8.002
City               8.003
Kms_Driven         0.000
Insurance_Valid    0.000
Service_History    0.000
Accidents          0.000
Tax_Paid           0.000
Number_of_Doors    0.000
Seats              0.000
Registration_Age   0.000
Price              0.000
dtype: float64


For numerical columns with missing data I have used MICE(Multiple Imputations by Chained Equations). The reason is because I wanted to use bayesian Imputation and since there are many columns with missing values MICE is the best as it inputs them at once.

In [35]:

imputer = IterativeImputer(random_state=0)

filled_data = imputer.fit_transform(df[["Mileage_kmpl","Engine_CC","Horsepower"]])
df[["Mileage_kmpl", "Engine_CC", "Horsepower"]] = filled_data

In [38]:
df = df.fillna(df.mode().iloc[0])
print(df.isnull().sum() / len(df) * 100)

Brand              0.000
Model              0.000
Year               0.000
Mileage_kmpl       0.000
Engine_CC          0.000
Horsepower         0.000
Fuel_Type          0.000
Transmission       0.000
Owner_Type         0.000
Color              0.000
City               0.000
Kms_Driven         0.000
Insurance_Valid    0.000
Service_History    0.000
Accidents          0.000
Tax_Paid           0.000
Number_of_Doors    0.000
Seats              0.000
Registration_Age   0.000
Price              0.000
dtype: float64


Now we try and see if our data contains any outliers.


In [39]:
df.describe()

,Year,Mileage_kmpl,Engine_CC,Horsepower,Kms_Driven,Insurance_Valid,Service_History,Accidents,Tax_Paid,Number_of_Doors,Seats,Registration_Age,Price
count,1005000.000,1005000.000,1005000.000,1005000.000,1005000.000,1005000.000,1005000.000,1005000.000,1005000.000,1005000.000,1005000.000,1005000.000,1005000.000
mean,2011.996,17.996,1506.716,120.572,202863.890,0.849,0.751,0.299,0.900,4.249,5.160,13.004,1012896.960
std,7.211,3.831,381.043,36.306,175370.541,0.358,0.433,0.547,0.300,0.699,0.914,7.211,1141644.936
min,2000.000,5.000,342.812,-18.510,5000.000,0.000,0.000,0.000,0.000,2.000,2.000,1.000,50000.000
25%,2006.000,15.571,1235.981,95.268,84555.000,1.000,1.000,0.000,1.000,4.000,5.000,7.000,343350.411
50%,2012.000,17.996,1503.631,120.359,166313.000,1.000,1.000,0.000,1.000,4.000,5.000,13.000,758908.320
75%,2018.000,20.426,1764.511,145.012,286380.000,1.000,1.000,1.000,1.000,5.000,5.000,19.000,1293475.466
max,2024.000,36.282,3362.023,290.250,4368000.000,1.000,1.000,5.000,1.000,5.000,7.000,25.000,37885997.533
